<a href="https://colab.research.google.com/github/dhruvsuri8106-code/ECON-3916---Statistical-and-Machine-Learning/blob/main/Lab%2013%20/%20Lab_13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

# Step 1: Ingestion and Naive Model
url = 'https://raw.githubusercontent.com/epleywin/ECON3916-Statistical-Machine-Learning/refs/heads/main/Zillow_California_2026_Hedonic.csv'
df = pd.read_csv(url)
df

,Property_Age,Distance_to_Tech_Hub,Sale_Price
0,77.5,38.1,684100.56
1,11.0,95.1,413634.22
2,47.7,73.5,456709.35
3,61.9,60.3,624533.95
4,100.8,16.4,870137.54
...,...,...,...
995,87.7,10.1,932592.35
996,21.2,91.8,412741.12
997,96.5,14.5,880901.56
998,20.1,95.1,396659.79


In [3]:
naive_model = smf.ols('Sale_Price ~ Property_Age', data=df).fit()
print(naive_model.summary())
print("\nNaive Age Coefficient:", naive_model.params['Property_Age'])

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.757
Model:                            OLS   Adj. R-squared:                  0.757
Method:                 Least Squares   F-statistic:                     3105.
Date:                Mon, 27 Apr 2026   Prob (F-statistic):          1.26e-308
Time:                        13:55:01   Log-Likelihood:                -12818.
No. Observations:                1000   AIC:                         2.564e+04
Df Residuals:                     998   BIC:                         2.565e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept     3.013e+05   7218.570     41.742   

In [4]:
# Step 3: FWL Theorem Manual Proof
# 3a: Partial out distance from Price
res_y_model = smf.ols('Sale_Price ~ Distance_to_Tech_Hub', data=df).fit()
df['Price_Residuals'] = res_y_model.resid
df['Price_Residuals']

,Price_Residuals
0,-56823.332740
1,19000.990249
2,-69149.815200
3,18481.157582
4,-2619.815668
...,...
995,21560.763160
996,-1940.516556
997,-3398.817768
998,2026.560249


In [5]:
# 3b: Partial out distance from Age
res_x_model = smf.ols('Property_Age ~ Distance_to_Tech_Hub', data=df).fit()
df['Age_Residuals'] = res_x_model.resid


In [6]:
# 3c: Regress Residuals on Residuals (-1 removes the intercept for exact mathematical matching)
fwl_model = smf.ols('Price_Residuals ~ Age_Residuals', data=df).fit()
print("\nFWL Isolated Age Coefficient:", fwl_model.params['Age_Residuals'])


FWL Isolated Age Coefficient: -2063.129216802138


In [7]:
"""
Multivariate OLS Regression — 3D Scatter Plot + Fitted Hyperplane
=================================================================
Predicts Sale_Price ~ Property_Age + Distance_to_Tech_Hub
Libraries: statsmodels, plotly, numpy, pandas
"""

import numpy as np
import pandas as pd
import statsmodels.api as sm
import plotly.graph_objects as go

# ── 0. Reproducible synthetic dataset ────────────────────────────────────────
np.random.seed(42)
n = 150

Property_Age        = np.random.uniform(1, 50, n)        # years old
Distance_to_Tech_Hub = np.random.uniform(0.5, 30, n)    # km from hub

# True DGP: newer homes and closer proximity → higher price
Sale_Price = (
    500_000
    - 3_500  * Property_Age
    - 8_000  * Distance_to_Tech_Hub
    + np.random.normal(0, 25_000, n)                     # noise term
)

df = pd.DataFrame({
    "Sale_Price":           Sale_Price,
    "Property_Age":         Property_Age,
    "Distance_to_Tech_Hub": Distance_to_Tech_Hub,
})

# ── 1. Fit the OLS model ──────────────────────────────────────────────────────
X = sm.add_constant(df[["Property_Age", "Distance_to_Tech_Hub"]])  # adds β₀ column
y = df["Sale_Price"]

model  = sm.OLS(y, X)
result = model.fit()

print(result.summary())

# ── 2. Extract coefficients from the statsmodels Results object ───────────────
# result.params is a pandas Series indexed by column names.
# We pull each coefficient by name so the mapping is explicit and error-proof.
beta_0   = result.params["const"]                  # intercept β₀
beta_age = result.params["Property_Age"]           # slope for Property_Age (β₁)
beta_dist = result.params["Distance_to_Tech_Hub"]  # slope for Distance (β₂)

print(f"\nExtracted Coefficients:")
print(f"  β₀ (intercept)          = {beta_0:,.2f}")
print(f"  β₁ (Property_Age)       = {beta_age:,.2f}")
print(f"  β₂ (Distance_to_Hub)    = {beta_dist:,.2f}")

# ── 3. Build the meshgrid for the regression hyperplane ───────────────────────
#
# Goal: evaluate ŷ = β₀ + β₁·X₁ + β₂·X₂ across a dense 2-D grid so Plotly
# can render it as a smooth surface.
#
# Step 3a — Create 1-D axis arrays spanning the observed range of each predictor.
#   np.linspace(min, max, 50) gives 50 evenly-spaced points per axis.
age_axis  = np.linspace(df["Property_Age"].min(),         df["Property_Age"].max(),  50)
dist_axis = np.linspace(df["Distance_to_Tech_Hub"].min(), df["Distance_to_Tech_Hub"].max(), 50)

# Step 3b — Expand the two 1-D axes into 2-D grids (50×50 matrices).
#   np.meshgrid returns two matrices:
#     AGE_GRID[i, j]  = age_axis[j]   → age value at column j, constant across rows
#     DIST_GRID[i, j] = dist_axis[i]  → distance value at row i, constant across cols
#   Together, every (i, j) cell represents one unique (age, distance) combination.
AGE_GRID, DIST_GRID = np.meshgrid(age_axis, dist_axis)

# Step 3c — Apply the linear equation element-wise across both grids.
#   Because AGE_GRID and DIST_GRID are both 50×50, numpy broadcasts the scalars
#   beta_0, beta_age, beta_dist automatically — no looping needed.
#   PRICE_SURFACE[i, j] = β₀ + β₁·AGE_GRID[i,j] + β₂·DIST_GRID[i,j]
PRICE_SURFACE = beta_0 + beta_age * AGE_GRID + beta_dist * DIST_GRID

# ── 4. Build the Plotly figure ────────────────────────────────────────────────
fig = go.Figure()

# 4a — Scatter trace: actual observed data points
fig.add_trace(go.Scatter3d(
    x=df["Property_Age"],
    y=df["Distance_to_Tech_Hub"],
    z=df["Sale_Price"],
    mode="markers",
    marker=dict(
        size=4,
        color=df["Sale_Price"],          # colour-encode the response variable
        colorscale="Viridis",
        colorbar=dict(title="Sale Price ($)", x=1.02),
        opacity=0.85,
        line=dict(width=0.3, color="white"),
    ),
    name="Observed Data",
    hovertemplate=(
        "<b>Property Age:</b> %{x:.1f} yrs<br>"
        "<b>Distance to Hub:</b> %{y:.1f} km<br>"
        "<b>Sale Price:</b> $%{z:,.0f}<extra></extra>"
    ),
))

# 4b — Surface trace: the OLS regression hyperplane
fig.add_trace(go.Surface(
    x=AGE_GRID,           # 50×50 matrix of age values
    y=DIST_GRID,          # 50×50 matrix of distance values
    z=PRICE_SURFACE,      # 50×50 matrix of predicted prices
    colorscale="RdBu",
    opacity=0.55,
    showscale=False,
    name="OLS Hyperplane",
    hovertemplate=(
        "<b>Age:</b> %{x:.1f} yrs<br>"
        "<b>Distance:</b> %{y:.1f} km<br>"
        "<b>Predicted Price:</b> $%{z:,.0f}<extra>OLS Plane</extra>"
    ),
))

# ── 5. Layout & annotations ───────────────────────────────────────────────────
coef_annotation = (
    f"β₀ = {beta_0:,.0f} | "
    f"β₁ (Age) = {beta_age:,.0f} | "
    f"β₂ (Distance) = {beta_dist:,.0f} | "
    f"R² = {result.rsquared:.3f}"
)

fig.update_layout(
    title=dict(
        text="Multivariate OLS Regression — Sale Price ~ Property Age + Distance to Tech Hub<br>"
             f"<sup>{coef_annotation}</sup>",
        x=0.5,
        font=dict(size=15),
    ),
    scene=dict(
        xaxis=dict(title="Property Age (years)", backgroundcolor="rgb(240,240,250)"),
        yaxis=dict(title="Distance to Tech Hub (km)", backgroundcolor="rgb(240,250,240)"),
        zaxis=dict(title="Sale Price ($)", tickformat="$,.0f", backgroundcolor="rgb(250,240,240)"),
        camera=dict(eye=dict(x=1.6, y=-1.6, z=0.9)),  # default viewing angle
        aspectmode="cube",
    ),
    legend=dict(x=0.01, y=0.95),
    margin=dict(l=0, r=0, t=100, b=0),
    height=720,
)

fig.show()
# fig.write_html("ols_3d_regression.html")   # ← uncomment to export

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.934
Model:                            OLS   Adj. R-squared:                  0.934
Method:                 Least Squares   F-statistic:                     1048.
Date:                Mon, 27 Apr 2026   Prob (F-statistic):           1.03e-87
Time:                        13:55:37   Log-Likelihood:                -1719.5
No. Observations:                 150   AIC:                             3445.
Df Residuals:                     147   BIC:                             3454.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                 5.015e+05 